# MiniCheck inside the grounding check? - `mcp-hallucination-checker`

**Question.** Does replacing the DeBERTa-v3-base NLI in the grounding check with **MiniCheck (flan-t5-large)** improve detection on the repo's tool-call benchmarks (`benchmarks/mcpevol.json`, and `benchmarks/mcpevol_v2.json` if present)?

**Background (from earlier runs).** On LLM-AggreFact, MiniCheck AUROC 0.821 vs windowed DeBERTa 0.744. In the MCPEvol ablation on the 35 text-based cases, whole-claim NLI-only (80%) beat the shipped span-level hybrid (57%). This notebook asks whether a stronger fact-checker, alone or combined with a structural veto, changes that picture on tool-call data.

| # | Variant | Decision rule (True = consistent) - all thresholds fixed in advance |
|---|---|---|
| 1 | **NLI-only (DeBERTa)** | whole hypothesis vs premise, `entailment.score_spans(premise, [hypothesis])` >= **0.40** (repo's `entailment_threshold`) |
| 2 | **MiniCheck-only** | MiniCheck(doc = premise, claim = hypothesis) >= **0.50** (MiniCheck's own default decision boundary). A threshold-sensitivity table is also shown and is clearly labelled **optimistic** (it is scored on the same data). |
| 3 | **Shipped hybrid** | `run_verification` (GED gate AND span-level NLI), repo code unchanged |
| 4 | **MiniCheck + asymmetric-GED veto** | MiniCheck >= **0.50** AND hypothesis-in-premise asymmetric GED (normalised) <= **0.35**. Asymmetric GED is copied from `benchmarks/ablation_local.py::asym_ged` (experimental, not part of the shipped server). |
| 5 | **Cascade (fast then slow)** | DeBERTa alone decides when its entailment is >= **0.90** (consistent) or <= **0.05** (hallucinated); otherwise MiniCheck (>= 0.50) decides. Reports the fraction of cases sent to MiniCheck and mean seconds per case. |

Schema/format cases are handled by the deterministic validator (`run_schema_alignment`), exactly as the repo's runner does; they are identical across variants and are only used to produce an overall (text + schema/format) number.

### Honest scope
* **MiniCheck is a published model** (Tang et al., EMNLP 2024) used off the shelf - nothing here is trained by us.
* **All thresholds are declared above and fixed before looking at results.** Nothing is tuned on these benchmarks. The only exception is the MiniCheck sensitivity table, which is explicitly "best-case / optimistic" and is not used for any headline number.
* **n is small.** `mcpevol.json` has only **35 text cases** (13+12 content, 3+2 negation, 3+2 role_swap). Confidence intervals are wide and most pairwise differences will not be statistically distinguishable. The benchmark is synthetic and was also used while developing the pipeline. **Treat everything as preliminary.**
* `mcpevol_v2.json` (if present) is a larger *generated* set; generated hypotheses may share templates and artefacts with the model-based scorers, so agreement on it is not independent evidence either.
* Latencies are per-case wall-clock on the Kaggle machine (batch size 1, GPU if available, warm-up excluded), measured in one process; the cascade and veto latencies are composed from separately timed components. Treat them as indicative.
* Positive class for precision / recall / specificity is **consistent** (same convention as `benchmarks/ablation_local.py`); specificity = fraction of hallucinated cases correctly flagged.

### Setup
1. Kaggle -> *Add Input* -> the dataset made from `kaggle/package_for_kaggle.py` (zip is auto-detected, exactly as in `ablation_mcpevol.ipynb`).
2. `benchmarks/mcpevol_v2.json` is **not** in that zip (the packager only includes `mcpevol.json`). To run the v2 part, attach it as an additional Kaggle input (any dataset containing a file called `mcpevol_v2.json`) or set `V2_PATH` below. If it is absent the notebook runs on `mcpevol.json` only.
3. Settings: **Internet = On** (pip install MiniCheck from GitHub, download `google/flan-t5-large` + MiniCheck weights, `dslim/bert-base-NER`, `cross-encoder/nli-deberta-v3-base`, spaCy model), **Accelerator = GPU T4**. No checkpoints are needed (small data).

## 1. Setup: dependencies, repo package, spaCy model, MiniCheck

In [ ]:
import os, sys, subprocess, importlib, glob, zipfile, json, time, math, re, random, contextlib, warnings, itertools, platform, functools
from pathlib import Path
warnings.filterwarnings("ignore")

def _ensure(pip_name, import_name=None):
    try:
        importlib.import_module(import_name or pip_name)
    except ImportError:
        print("pip install", pip_name)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

for _pip, _imp in [("numpy", None), ("pandas", None), ("scipy", None), ("networkx", None), ("pydantic", None),
                   ("pyyaml", "yaml"), ("spacy", None), ("torch", None), ("transformers", None),
                   ("sentencepiece", None), ("protobuf", "google.protobuf"), ("nltk", None), ("tqdm", None)]:
    _ensure(_pip, _imp)

import numpy as np, pandas as pd, torch, transformers, spacy
print("python", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__, "| spacy", spacy.__version__)
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

# ---- locate the repo package (dataset made by kaggle/package_for_kaggle.py) -------------------------
def _has_pkg(d):
    return (Path(d) / "server" / "pipeline" / "ged.py").exists()

def locate_package():
    env = os.environ.get("MCP_PKG_DIR")
    if env and _has_pkg(env):
        return Path(env)
    for ged in sorted(glob.glob("/kaggle/input/**/server/pipeline/ged.py", recursive=True)):
        return Path(ged).parents[2]
    for z in sorted(glob.glob("/kaggle/input/**/*.zip", recursive=True)):
        try:
            with zipfile.ZipFile(z) as zf:
                if any(n.endswith("server/pipeline/ged.py") for n in zf.namelist()):
                    dest = Path("/kaggle/working/pkg"); dest.mkdir(parents=True, exist_ok=True)
                    zf.extractall(dest)
                    for ged in dest.rglob("server/pipeline/ged.py"):
                        return ged.parents[2]
        except zipfile.BadZipFile:
            pass
    for d in [Path.cwd(), *Path.cwd().parents]:
        if _has_pkg(d):
            return d
    raise FileNotFoundError("Repo package not found. Upload the zip from kaggle/package_for_kaggle.py as a Kaggle dataset and attach it (Add Input).")

PKG_ROOT = locate_package()
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))
print("package root:", PKG_ROOT)

# ---- spaCy model: same preference order as server/pipeline/srl.py (trf, falls back to sm) --------------
USE_TRF = True
SPACY_MODELS = ["en_core_web_trf", "en_core_web_sm"] if USE_TRF else ["en_core_web_sm"]
def _try_load(name):
    try:
        spacy.load(name); return True
    except Exception:
        return False
if not any(_try_load(m) for m in SPACY_MODELS):
    for m in SPACY_MODELS:
        print("installing spaCy model", m, "(needs Internet = On)")
        r = subprocess.run([sys.executable, "-m", "spacy", "download", m], capture_output=True, text=True)
        importlib.invalidate_caches()
        if _try_load(m): break
        print(r.stdout[-400:], r.stderr[-400:])
assert any(_try_load(m) for m in SPACY_MODELS), "No spaCy model could be loaded (Internet On? restart kernel after install and re-run)."

# ---- MiniCheck (published model; install command from the official README) -----------------------------
MINICHECK_PIP = "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@main"
try:
    import minicheck  # noqa: F401
except ImportError:
    print("Installing MiniCheck (needs Internet = On) ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", MINICHECK_PIP])
import nltk
for _r in ("punkt", "punkt_tab"):          # MiniCheck chunks documents with nltk.sent_tokenize
    try: nltk.download(_r, quiet=True)
    except Exception as _e: print(f"nltk.download({_r!r}) failed (continuing): {_e}")
nltk.sent_tokenize("Smoke test one. Smoke test two.")
print("setup done")

## 2. Configuration (all thresholds fixed in advance)

In [ ]:
SEED = 42
N_BOOT = 2000

# ---- pre-declared decision thresholds; NOTHING below is tuned on the benchmarks ----------------------
ENT_THR_NLI   = 0.40    # DeBERTa NLI whole-claim entailment (== configs/thresholds.yaml entailment_threshold; asserted below)
MC_THR        = 0.50    # MiniCheck's own default decision boundary
ASYM_GED_THR  = 0.35    # same value as the shipped ged_threshold, applied to the (different-scale, experimental) asymmetric GED
CASCADE_HI    = 0.90    # DeBERTa entailment >= this -> consistent without MiniCheck
CASCADE_LO    = 0.05    # DeBERTa entailment <= this -> hallucinated without MiniCheck
MC_SENSITIVITY_GRID = [round(x, 2) for x in np.arange(0.05, 0.951, 0.05)]   # reporting only, labelled OPTIMISTIC

MINICHECK_MODEL = "flan-t5-large"
MINICHECK_BATCH = 4
MINICHECK_CACHE = "/kaggle/temp/minicheck_cache" if os.path.isdir("/kaggle/temp") else ("/kaggle/working/minicheck_cache" if os.path.isdir("/kaggle/working") else "./minicheck_cache")
USE_GPU_FOR_DEBERTA = True            # same models/logic, only the device changes (so latencies are comparable with MiniCheck on GPU)

V2_PATH = os.environ.get("MCPEVOL_V2")      # optional explicit path to mcpevol_v2.json
SMOKE_N = int(os.environ["MC_SMOKE_N"]) if os.environ.get("MC_SMOKE_N") else None   # local smoke tests only: keep first N cases per dataset split

OUT_DIR = Path(os.environ.get("MC_OUT_DIR") or ("/kaggle/working" if os.path.isdir("/kaggle/working") else "./minicheck_out"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = OUT_DIR / "minicheck_pipeline_results.json"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
TEXT_TYPES = ("content", "negation", "role_swap")
TYPED_TYPES = ("schema", "format")
print("output:", RESULTS_PATH)

## 3. Repo imports, DeBERTa on GPU, MiniCheck load

In [ ]:
from server import config as repo_config
from server.pipeline import entailment, srl, ged as ged_mod
from server.pipeline.graph_builder import build_semantic_graph
from server.tools import verify_logic
from server.tools.schema import run_schema_alignment
from scipy.optimize import linear_sum_assignment

GED_THR = repo_config.GED_THRESHOLD
assert abs(repo_config.ENTAILMENT_THRESHOLD - ENT_THR_NLI) < 1e-9, \
    f"repo entailment_threshold={repo_config.ENTAILMENT_THRESHOLD} != declared {ENT_THR_NLI}; fix the declaration, do not tune"
print(f"repo thresholds: ged={GED_THR} entailment={repo_config.ENTAILMENT_THRESHOLD}")

if USE_GPU_FOR_DEBERTA and torch.cuda.is_available():
    from transformers import pipeline as _hf_pipeline
    entailment._load_pipeline.cache_clear()
    entailment._load_pipeline = functools.lru_cache(maxsize=1)(lambda: _hf_pipeline(
        "text-classification", model=entailment.MODEL_NAME, top_k=None, truncation=True, device=0))
    srl._load_ner.cache_clear()
    srl._load_ner = functools.lru_cache(maxsize=1)(lambda: _hf_pipeline(
        "token-classification", model="dslim/bert-base-NER", aggregation_strategy="simple", device=0))
    print("DeBERTa NLI + BERT NER -> GPU")
else:
    print("DeBERTa NLI + BERT NER on CPU")

from minicheck.minicheck import MiniCheck
os.makedirs(MINICHECK_CACHE, exist_ok=True)
mc_scorer = MiniCheck(model_name=MINICHECK_MODEL, batch_size=MINICHECK_BATCH, cache_dir=MINICHECK_CACHE)

def mc_prob(doc, claim):
    """MiniCheck probability that `claim` is supported by `doc` (README API: score() returns (labels, probs, ...))."""
    out = mc_scorer.score(docs=[doc], claims=[claim])
    return float(out[1][0])

# README example sanity check (expect roughly [1, 0])
_d = "A group of students gather in the school library to study for their upcoming final exams."
_lab, _p = mc_scorer.score(docs=[_d, _d], claims=["The students are preparing for an examination.", "The students are on vacation."])[:2]
print("MiniCheck README smoke test (expect ~[1, 0]):", list(_lab), [round(float(x), 4) for x in _p])

## 4. Experimental asymmetric GED (copied from `benchmarks/ablation_local.py::asym_ged`)
Cost of embedding the hypothesis graph into the premise graph: no symmetry swap, hypothesis-side cost only, no premise-extra penalty, no value bonus. Copied verbatim (function body), repo file untouched. It is experimental and not part of the shipped server.

In [ ]:
def asym_ged(G_hyp, G_prem):
    """Experimental: cost of embedding the hypothesis graph into the premise graph."""
    hn, pn = list(G_hyp.nodes(data=True)), list(G_prem.nodes(data=True))
    n, m = len(hn), len(pn)
    if n == 0:
        return 0.0, 0.0
    if m == 0:
        return float(n), 1.0
    cost = np.array([[ged_mod._node_subst_cost(a, aa, b, bb) for b, bb in pn] for a, aa in hn])
    rows, cols = linear_sum_assignment(cost)
    total = float(cost[rows, cols].sum()) + max(0, n - m)  # unmatched hypothesis nodes cost 1
    mapping = {hn[i][0]: pn[j][0] for i, j in zip(rows, cols)}
    for u, v, d in G_hyp.edges(data=True):
        mu, mv = mapping.get(u), mapping.get(v)
        if mu is not None and mv is not None and G_prem.has_edge(mu, mv):
            total += ged_mod._edge_subst_cost(d, G_prem[mu][mv])
        else:
            total += 1.0
    denom = n + G_hyp.number_of_edges()
    return total, max(0.0, min(1.0, total / denom if denom else 0.0))

## 5. Variant logic, metrics, bootstrap, McNemar

In [ ]:
VARIANTS = ["1 NLI-only (DeBERTa)", "2 MiniCheck-only", "3 shipped hybrid", "4 MiniCheck + asym-GED veto", "5 cascade DeBERTa->MiniCheck"]

def decide(nli, mc, asym, hybrid):
    """Pure decision logic from cached scores (True = consistent). Thresholds are the pre-declared constants.
    Returns (predictions dict, sent_to_minicheck flag for the cascade)."""
    cascade_decided = (nli >= CASCADE_HI) or (nli <= CASCADE_LO)
    return {
        "1 NLI-only (DeBERTa)":          bool(nli >= ENT_THR_NLI),
        "2 MiniCheck-only":              bool(mc >= MC_THR),
        "3 shipped hybrid":              bool(hybrid),
        "4 MiniCheck + asym-GED veto":   bool((mc >= MC_THR) and (asym <= ASYM_GED_THR)),
        "5 cascade DeBERTa->MiniCheck":  bool((nli >= CASCADE_HI) if cascade_decided else (mc >= MC_THR)),
    }, (not cascade_decided)

def metrics(y, p):
    """y, p: bool, True = consistent = positive class (same convention as benchmarks/ablation_local.py)."""
    y, p = np.asarray(y, bool), np.asarray(p, bool)
    tp, fp, tn, fn = int((p & y).sum()), int((p & ~y).sum()), int((~p & ~y).sum()), int((~p & y).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    spec = tn / (tn + fp) if tn + fp else 0.0
    return dict(n=len(y), accuracy=(tp + tn) / len(y) if len(y) else float("nan"), balanced_accuracy=(rec + spec) / 2,
                precision=prec, recall=rec, specificity=spec, tp=tp, fp=fp, tn=tn, fn=fn)

def boot_ci(y, p, key):
    rng = np.random.default_rng(SEED)
    y, p = np.asarray(y, bool), np.asarray(p, bool); vals = []
    for _ in range(N_BOOT):
        i = rng.integers(0, len(y), len(y))
        if key == "balanced_accuracy" and (y[i].all() or (~y[i]).all()):
            continue
        vals.append(metrics(y[i], p[i])[key])
    if not vals:      # e.g. only one class present -> CI undefined
        return [None, None]
    return [round(float(np.percentile(vals, 2.5)), 3), round(float(np.percentile(vals, 97.5)), 3)]

def mcnemar_exact(y, pa, pb):
    ca, cb = np.asarray(pa) == np.asarray(y), np.asarray(pb) == np.asarray(y)
    b, c = int((ca & ~cb).sum()), int((~ca & cb).sum()); n = b + c
    if n == 0: return dict(only_a_correct=b, only_b_correct=c, p=1.0)
    k = min(b, c)
    return dict(only_a_correct=b, only_b_correct=c, p=round(min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n), 4))

def auroc(y, s):
    from scipy.stats import rankdata
    y = np.asarray(y, bool); s = np.asarray(s, float); n1, n0 = int(y.sum()), int((~y).sum())
    if n1 == 0 or n0 == 0: return float("nan")
    return float((rankdata(s)[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

## 6. Load the benchmarks
`mcpevol.json` comes from the repo package. `mcpevol_v2.json` is optional (see Setup). Schema: `id, hypothesis, premise, label (True = consistent), type`, optionally `agent_args` / `tool_schema`.

In [ ]:
def find_v2():
    cands = [V2_PATH] if V2_PATH else []
    cands += [PKG_ROOT / "benchmarks" / "mcpevol_v2.json"]
    cands += [Path(p) for p in sorted(glob.glob("/kaggle/input/**/mcpevol_v2.json", recursive=True))]
    cands += [Path.cwd() / "benchmarks" / "mcpevol_v2.json"]
    for c in cands:
        if c and Path(c).exists():
            return Path(c)
    return None

def load_records(path):
    recs = json.loads(Path(path).read_text(encoding="utf-8"))
    out = []
    for r in recs:
        r = dict(r); r["type"] = r.get("type") or "content"; r["label"] = bool(r["label"]); out.append(r)
    return out

DATASETS = {"mcpevol": load_records(PKG_ROOT / "benchmarks" / "mcpevol.json")}
_v2 = find_v2()
if _v2:
    DATASETS["mcpevol_v2"] = load_records(_v2); print("found v2:", _v2)
else:
    print("mcpevol_v2.json not found -> running on mcpevol.json only")
for name, recs in DATASETS.items():
    text = [r for r in recs if r["type"] not in TYPED_TYPES]; other = [r for r in recs if r["type"] in TYPED_TYPES]
    unk = [r for r in recs if r["type"] not in TEXT_TYPES + TYPED_TYPES]
    print(f"{name}: {len(recs)} cases | text {len(text)} ({sum(r['label'] for r in text)} consistent / {sum(not r['label'] for r in text)} hallucinated)"
          f" | schema/format {len(other)}")
    print("   by type:", pd.Series([r['type'] for r in recs]).value_counts().to_dict())
    if unk: print("   NOTE: types outside text/schema/format are treated as text cases:", sorted({r['type'] for r in unk}))

## 7. Per-case scoring and timing
For every text case: DeBERTa whole-claim entailment, MiniCheck probability, asymmetric GED, and the shipped `run_verification`. Each is timed separately (batch size 1, after a warm-up, so model-load time is excluded). The variants are then derived from these cached scores, so no model is run twice. Latency per variant: NLI-only = DeBERTa time; MiniCheck-only = MiniCheck time; shipped hybrid = `run_verification` time; veto = MiniCheck + graph building + asymmetric GED; cascade = DeBERTa + (MiniCheck if routed).

In [ ]:
from tqdm.auto import tqdm

def sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()

def timed(fn, *a):
    sync(); t = time.perf_counter(); r = fn(*a); sync()
    return r, time.perf_counter() - t

# warm-up (model init, CUDA kernels, spaCy) - excluded from timings
_wh, _wp = "The file report.txt was uploaded by Alice.", "Alice uploaded the file report.txt to the shared folder on Monday."
entailment.score_spans(_wp, [_wh]); mc_prob(_wp, _wh); verify_logic.run_verification(_wh, _wp, GED_THR)
_g1, _g2 = build_semantic_graph(_wh), build_semantic_graph(_wp); asym_ged(_g1, _g2)

def score_case(r):
    h, p = r["hypothesis"], r["premise"]
    t = {}
    def graphs():
        return build_semantic_graph(h), build_semantic_graph(p)
    (Gh, Gp), t["graph"] = timed(graphs)
    (_, norm_a), t["asym"] = timed(asym_ged, Gh, Gp)
    nli, t["nli"] = timed(lambda: entailment.score_spans(p, [h])[0][1] if h.strip() else 0.0)
    mc, t["mc"] = timed(mc_prob, p, h)
    hyb, t["hybrid"] = timed(lambda: bool(verify_logic.run_verification(h, p, GED_THR).is_consistent))
    preds, to_mc = decide(float(nli), float(mc), float(norm_a), hyb)
    lat = {"1 NLI-only (DeBERTa)": t["nli"], "2 MiniCheck-only": t["mc"], "3 shipped hybrid": t["hybrid"],
           "4 MiniCheck + asym-GED veto": t["mc"] + t["graph"] + t["asym"],
           "5 cascade DeBERTa->MiniCheck": t["nli"] + (t["mc"] if to_mc else 0.0)}
    return dict(id=r["id"], type=r["type"], label=bool(r["label"]), nli=float(nli), mc=float(mc), asym_ged=float(norm_a),
                hybrid=hyb, sent_to_minicheck=bool(to_mc), timings=t, pred=preds, latency=lat)

def score_text_cases(recs):
    text = [r for r in recs if r["type"] not in TYPED_TYPES]
    if SMOKE_N: text = text[:SMOKE_N]
    return [score_case(r) for r in tqdm(text)]

def score_typed_cases(recs):
    """Deterministic validator, as in benchmarks/ablation_local.py and the repo runner."""
    sch = [r for r in recs if r["type"] in TYPED_TYPES]
    if SMOKE_N: sch = sch[:SMOKE_N]
    out = []
    for r in sch:
        res, dt = timed(lambda: bool(run_schema_alignment(r.get("agent_args", {}), r.get("tool_schema"), threshold=GED_THR).is_aligned))
        out.append(dict(id=r["id"], type=r["type"], label=bool(r["label"]), pred=res, latency=dt))
    return out

SCORED = {}
for name, recs in DATASETS.items():
    t0 = time.time()
    SCORED[name] = dict(text=score_text_cases(recs), typed=score_typed_cases(recs))
    print(f"{name}: scored {len(SCORED[name]['text'])} text + {len(SCORED[name]['typed'])} schema/format cases in {time.time()-t0:.0f}s")

## 8. Results
Headline numbers are at the **pre-declared** thresholds. Positive class = consistent. `bal_acc` = (recall + specificity) / 2. CIs are percentile bootstrap (2000 resamples of cases; they do not capture threshold or model variance). McNemar is exact, two-sided, per-case correctness, **not** corrected for the 10 pairwise comparisons.

In [ ]:
def analyse(name):
    S = SCORED[name]; T = S["text"]; X = S["typed"]
    y = np.array([c["label"] for c in T], bool); types = np.array([c["type"] for c in T])
    P = {v: np.array([c["pred"][v] for c in T], bool) for v in VARIANTS}
    A = {"dataset": name, "n_text": len(T), "n_text_consistent": int(y.sum()), "n_text_hallucinated": int((~y).sum()), "n_schema_format": len(X),
         "thresholds": dict(nli=ENT_THR_NLI, minicheck=MC_THR, asym_ged=ASYM_GED_THR, cascade_hi=CASCADE_HI, cascade_lo=CASCADE_LO, tuned_on_benchmark=False)}
    A["text_variants"] = {}
    for v in VARIANTS:
        m = metrics(y, P[v])
        m["accuracy_95ci"] = boot_ci(y, P[v], "accuracy"); m["balanced_accuracy_95ci"] = boot_ci(y, P[v], "balanced_accuracy")
        lat = np.array([c["latency"][v] for c in T])
        m["latency_s_mean"] = round(float(lat.mean()), 4); m["latency_s_median"] = round(float(np.median(lat)), 4)
        A["text_variants"][v] = m
    A["by_type"] = {}
    for t in sorted(set(types), key=lambda t: (t not in TEXT_TYPES, t)):
        idx = np.where(types == t)[0]
        A["by_type"][t] = {"n": int(len(idx)), **{v: round(float(metrics(y[idx], P[v][idx])["accuracy"]), 3) for v in VARIANTS}}
    A["mcnemar_exact"] = {f"{a} vs {b}": mcnemar_exact(y, P[a], P[b]) for a, b in itertools.combinations(VARIANTS, 2)}
    sent = np.array([c["sent_to_minicheck"] for c in T], bool)
    cv = VARIANTS[4]
    A["cascade"] = dict(fraction_sent_to_minicheck=round(float(sent.mean()), 3), n_sent=int(sent.sum()), n=len(T),
                        mean_s_per_case=A["text_variants"][cv]["latency_s_mean"],
                        mean_s_per_case_minicheck_only=A["text_variants"]["2 MiniCheck-only"]["latency_s_mean"],
                        accuracy_on_auto_decided=round(float(metrics(y[~sent], P[cv][~sent])["accuracy"]), 3) if (~sent).any() else None,
                        accuracy_on_sent=round(float(metrics(y[sent], P[cv][sent])["accuracy"]), 3) if sent.any() else None)
    A["auroc"] = dict(deberta_nli=round(auroc(y, [c["nli"] for c in T]), 3), minicheck=round(auroc(y, [c["mc"] for c in T]), 3),
                      one_minus_asym_ged=round(auroc(y, [-c["asym_ged"] for c in T]), 3))
    A["minicheck_threshold_sensitivity_OPTIMISTIC"] = []
    for thr in MC_SENSITIVITY_GRID:
        pm = np.array([c["mc"] >= thr for c in T]); m = metrics(y, pm)
        A["minicheck_threshold_sensitivity_OPTIMISTIC"].append(dict(threshold=thr, accuracy=round(m["accuracy"], 3), balanced_accuracy=round(m["balanced_accuracy"], 3),
                                                                    precision=round(m["precision"], 3), recall=round(m["recall"], 3), specificity=round(m["specificity"], 3)))
    if X:
        yx = np.array([c["label"] for c in X], bool); px = np.array([c["pred"] for c in X], bool)
        A["schema_format_validator"] = metrics(yx, px)
        yall = np.concatenate([y, yx]); A["overall"] = {}
        for v in VARIANTS:
            pa = np.concatenate([P[v], px]); m = metrics(yall, pa)
            m["accuracy_95ci"] = boot_ci(yall, pa, "accuracy"); A["overall"][v] = m
        A["overall_n"] = int(len(yall))
    A["predictions"] = [{"id": c["id"], "type": c["type"], "label": c["label"], "nli": round(c["nli"], 4), "minicheck": round(c["mc"], 4),
                         "asym_ged": round(c["asym_ged"], 4), "sent_to_minicheck": c["sent_to_minicheck"],
                         **{v: bool(c["pred"][v]) for v in VARIANTS}, "latency_s": {v: round(c["latency"][v], 4) for v in VARIANTS}} for c in T]
    return A

ANALYSIS = {}
for name in SCORED:
    ANALYSIS[name] = A = analyse(name)
    print("=" * 110)
    print(f"{name}: {A['n_text']} text cases ({A['n_text_consistent']} consistent / {A['n_text_hallucinated']} hallucinated) | thresholds fixed: "
          f"DeBERTa>={ENT_THR_NLI}, MiniCheck>={MC_THR}, asymGED<={ASYM_GED_THR}, cascade {CASCADE_LO}/{CASCADE_HI}")
    rows = []
    for v in VARIANTS:
        m = A["text_variants"][v]
        rows.append({"variant": v, "acc": m["accuracy"], "acc 95% CI": m["accuracy_95ci"], "bal_acc": m["balanced_accuracy"], "bal_acc 95% CI": m["balanced_accuracy_95ci"],
                     "precision": m["precision"], "recall": m["recall"], "specificity": m["specificity"],
                     "TP/FP/TN/FN": f"{m['tp']}/{m['fp']}/{m['tn']}/{m['fn']}", "s/case mean": m["latency_s_mean"], "s/case median": m["latency_s_median"]})
    print(pd.DataFrame(rows).set_index("variant").round(3).to_string())
    print("\nPer-type accuracy:"); print(pd.DataFrame(A["by_type"]).T.to_string())
    print("\nCascade:", json.dumps(A["cascade"]))
    print("AUROC (consistent = positive):", json.dumps(A["auroc"]))
    print("\nExact McNemar (only_a_correct / only_b_correct / p; uncorrected):")
    print(pd.DataFrame(A["mcnemar_exact"]).T.to_string())
    print("\nMiniCheck threshold sensitivity - OPTIMISTIC, scored on the same cases, NOT a result:")
    print(pd.DataFrame(A["minicheck_threshold_sensitivity_OPTIMISTIC"]).to_string(index=False))
    if "overall" in A:
        print(f"\nOverall incl. {A['n_schema_format']} schema/format cases via the deterministic validator "
              f"(validator acc {A['schema_format_validator']['accuracy']:.3f}); n={A['overall_n']}:")
        print(pd.DataFrame({v: {"accuracy": A["overall"][v]["accuracy"], "95% CI": A["overall"][v]["accuracy_95ci"]} for v in VARIANTS}).T.to_string())

## 9. Error analysis (cases where at least one variant is wrong)

In [ ]:
for name in SCORED:
    T = SCORED[name]["text"]
    rows = []
    for c in T:
        wrong = [v for v in VARIANTS if c["pred"][v] != c["label"]]
        if wrong:
            rows.append({"id": c["id"], "type": c["type"], "truth": "consistent" if c["label"] else "HALLUC", "nli": round(c["nli"], 3),
                         "minicheck": round(c["mc"], 3), "asym_ged": round(c["asym_ged"], 3),
                         "wrong": ",".join(v.split()[0] for v in wrong)})
    print(f"\n{name}: {len(rows)}/{len(T)} cases wrong under at least one variant (wrong = variant numbers 1-5)")
    if rows: print(pd.DataFrame(rows).to_string(index=False))

## 10. Save results

In [ ]:
RESULTS = {
    "what": "MiniCheck (flan-t5-large) vs DeBERTa NLI in the grounding check on MCPEvol text cases. Published model used off the shelf; all thresholds fixed in advance; n small; preliminary.",
    "environment": {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__, "spacy": spacy.__version__,
                    "cuda": torch.cuda.is_available(), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                    "deberta_model": entailment.MODEL_NAME, "minicheck_model": MINICHECK_MODEL, "minicheck_pip": MINICHECK_PIP,
                    "minicheck_batch_size": MINICHECK_BATCH, "package_root": str(PKG_ROOT), "v2_path": str(_v2) if _v2 else None},
    "config": {"deberta_threshold": ENT_THR_NLI, "minicheck_threshold": MC_THR, "asym_ged_threshold": ASYM_GED_THR, "repo_ged_threshold": GED_THR,
               "cascade_hi": CASCADE_HI, "cascade_lo": CASCADE_LO, "n_boot": N_BOOT, "seed": SEED, "smoke_n": SMOKE_N,
               "positive_class": "consistent", "tuned_on_benchmark": False},
    "datasets": ANALYSIS,
}
RESULTS_PATH.write_text(json.dumps(RESULTS, indent=2, default=float), encoding="utf-8")
print("saved", RESULTS_PATH)

## 11. How to read this
* Compare variants only through the McNemar table and the CIs: with 35 text cases a gap of a few cases is not evidence. Any recommendation should rest on the larger `mcpevol_v2` run **and** the earlier AggreFact AUROC result, not on `mcpevol.json` alone.
* MiniCheck-only vs NLI-only is the cleanest comparison (one threshold each, both fixed in advance). Variant 4 adds a structural veto that can only lower recall and raise specificity relative to variant 2; variant 5 is an engineering trade-off (speed vs accuracy) - judge it by the fraction sent to MiniCheck, the seconds per case, and accuracy on the auto-decided subset.
* The MiniCheck sensitivity table is the best case under hindsight; do not quote its maximum as a result.
* Latencies include Python overhead at batch size 1 and exclude warm-up/model loading.
* The benchmark texts are short, synthetic and template-like; they differ from the long-document LLM-AggreFact setting in which MiniCheck was trained and evaluated.